# Recovering audio from old 78rpm records

<img src="img/phonograph.png" style="float: left; height: 300px; margin: 10px 0 0 20px;"/>
<img src="img/player.png" style="float: left; height: 200px; margin: 80px 0 0 200px;"/>


In [1]:
import numpy as np
import matplotlib.pyplot as plt
import IPython
import scipy.signal as sp
from scipy.io import wavfile

plt.rcParams["figure.figsize"] = (12,3)

## The audio

This is how the record, played on a 78rpm phonograph, would sound

In [2]:
rate, audio78 = wavfile.read('data/sandman.wav')
IPython.display.Audio(audio78, rate=rate)

If we play it on a 33rpm turntable it will sound like this:

In [14]:
_, audio33 = wavfile.read('data/sandman33.wav')
IPython.display.Audio(audio33, rate=rate)

## Speeding up the audio

As a first attempt, let's simply downsample the "slow" audio by a factor of 2; in Python, downsampling can be easily implemented via slicing

In [15]:
IPython.display.Audio(audio33[::2], rate=rate)

It sounds closer to the original but there are 2 problems:
 * the pitch is not quite right, still a bit high
 * audio artefacts due to raw downsampling

The right speed-up factor is the ratio of turntable speeds:

In [16]:
78/33

2.3636363636363638

which can be reduced to a ratio of coprime integers:

In [17]:
26/11

2.3636363636363638

## Fractional up/downsampling

in Python we can perform upsampling using the Kronecker product; eg, 4-upsampling:

In [18]:
np.kron([1,2,3], [1, 0, 0, 0])

array([1, 0, 0, 0, 2, 0, 0, 0, 3, 0, 0, 0])

and we can combine that directly with slicing if up- and downsampling factors are coprime; in this example we upsample by 2 and downsample by 5:

In [19]:
y = np.kron(audio33, [1, 0])[::5]

In [20]:
IPython.display.Audio(y, rate=rate)

This sounds almost like what we want since $26/11 \approx 5/2$; let's now do it properly

In [21]:
UP = 11
DOWN = 26

y = np.kron(audio33, np.r_[1, np.zeros(UP-1)])[::DOWN]

In [22]:
IPython.display.Audio(y, rate=rate)

with such a large downsampling factor we cannot skip the lowpass filter! Let's add it 

In [12]:
b, a = sp.butter(4, 1/DOWN)

xu = np.kron(audio33, np.r_[1, np.zeros(10)])
xi = sp.lfilter(b, a, xu)
y = xi[::DOWN]

IPython.display.Audio(y, rate=rate)

which is the same as playing the original record at 78rpm


In [13]:
IPython.display.Audio(audio78, rate=rate)